# agentic_alpha_meta_label_probability_gate_search_v1

Diagnostic R-G-E-V-D card for a meta-labeling probability gate branch. It maps external agentic-alpha workflow guidance plus meta-labeling references into a qlib-native SOXL/SOXX/VIX walk-forward probability gate, then validates raw numeric passes against exposure and trade-count sanity gates.

## R-G-E-V-D Route

- Research/Retrieve: `agentic-alpha` R-G-E-V-D, ML-for-trading meta-labeling examples, Hudson & Thames meta-labeling, and arXiv portfolio meta-learning.
- Generate: train annual walk-forward meta-label classifiers on primary SOXL side signals.
- Evaluate/Validate: compare every probability/scale grid row to `qlib_paper_sim_v1` and require MaxDD >= -20%.
- Distill: raw numeric passes are not promotable unless they also pass `max_exposure <= 10x` and `trade_count >= 20`.

In [1]:

from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.factor import (
    build_meta_label_probability_frame,
    compact_meta_label_probability_summary,
    evaluate_meta_label_probability_candidates,
)
from src.factor.meta_label_probability_gate import META_LABEL_MODEL_FAMILIES, META_LABEL_SOURCE_FIELDS
from src.workflow import load_agentic_alpha_backtest_metric

EXPERIMENT = 'agentic_alpha_meta_label_probability_gate_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
PRODUCTION_CARD = ROOT / 'production' / 'qlib_paper_sim_v1'
START = '2019-01-02'
END = '2026-06-09'
MAX_DRAWDOWN_FLOOR = -20.0
MAX_EXPOSURE_SANITY = 10.0
MIN_TRADE_COUNT_SANITY = 20
TRANSACTION_COST = 0.0002

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: diagnostic_search
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
segments:
  train_start: 2010-03-11
  test:
    start: {START}
    end: {END}
rgevd:
  branch: meta_label_probability_gate
  stages:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  require_return_above_baseline: true
  max_exposure_sanity: {MAX_EXPOSURE_SANITY}
  min_trade_count_sanity: {MIN_TRADE_COUNT_SANITY}
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')

external_sources = pd.DataFrame(
    [
        {
            'source_type': 'github',
            'title': 'agentic-alpha R-G-E-V-D workflow',
            'url': 'https://github.com/JacobDu/agentic-alpha',
            'extracted_signal': 'Use Research/Retrieve/Generate/Evaluate/Validate/Distill loops and close each branch with evidence.',
            'retrieval_status': 'workflow_reference',
        },
        {
            'source_type': 'github',
            'title': 'Machine Learning for Trading meta-labeling examples',
            'url': 'https://github.com/stefan-jansen/machine-learning-for-trading',
            'extracted_signal': 'Use a secondary classifier to gate primary side signals rather than sizing every raw signal equally.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Meta Labeling: A Toy Example',
            'url': 'https://hudsonthames.org/meta-labeling-a-toy-example/',
            'extracted_signal': 'Meta-labeling separates side generation from probability-of-success validation.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'arxiv',
            'title': 'Meta-learning for portfolio management',
            'url': 'https://arxiv.org/html/2505.03659v2',
            'extracted_signal': 'Meta-learning can adapt allocation decisions, but validation must guard against sparse high-confidence overfit.',
            'retrieval_status': 'web_retrieved',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)

baseline = load_agentic_alpha_backtest_metric(PRODUCTION_CARD, window='test')
print(f'baseline_return={baseline.return_pct:.4f}, baseline_maxdd={baseline.max_drawdown_pct:.4f}')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


baseline_return=6570.0233, baseline_maxdd=-85.2317


In [2]:
qlib.init(provider_uri=PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
raw = D.features(
    ['SOXL', 'SOXX'],
    ['$open', '$high', '$low', '$close', '$volume', '$vix'],
    start_time='2010-03-11',
    end_time='2026-06-10',
    freq='day',
)
features = build_meta_label_probability_frame(raw, start=START, end=END, min_history=60)
metrics = evaluate_meta_label_probability_candidates(
    features,
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
    event_cols=['base_trend', 'base_dip', 'base_calendar', 'base_any'],
    models=['logit', 'hgb'],
    thresholds=[0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9],
    scales=[1, 2, 3, 4, 6, 10, 20, 40, 60, 80, 100],
    size_modes=['binary', 'probability'],
    train_min_events=80,
    max_exposure_sanity=MAX_EXPOSURE_SANITY,
    min_trade_count_sanity=MIN_TRADE_COUNT_SANITY,
    cost_per_turnover=TRANSACTION_COST,
)
search_summary = compact_meta_label_probability_summary(
    metrics,
    branches=['meta_label_probability_gate'],
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
    max_exposure_sanity=MAX_EXPOSURE_SANITY,
    min_trade_count_sanity=MIN_TRADE_COUNT_SANITY,
)
search_summary['baseline'] = {
    'name': baseline.name,
    'return_pct': float(baseline.return_pct),
    'max_drawdown_pct': float(baseline.max_drawdown_pct),
    'benchmark_return_pct': float(baseline.benchmark_return_pct),
    'recorder_id': baseline.recorder_id,
}
search_summary['external_source_counts'] = external_sources['source_type'].value_counts().sort_index().to_dict()
search_summary['feature_evidence'] = {
    'feature_rows': int(len(features)),
    'metric_rows': int(len(metrics)),
    'source_coverage_start': features.attrs.get('source_coverage_start'),
    'source_coverage_end': features.attrs.get('source_coverage_end'),
    'test_start': START,
    'test_end': END,
    'primary_signals': ['base_trend', 'base_dip', 'base_calendar', 'base_any'],
}

metrics.to_csv(CARD / 'agentic_alpha_meta_label_probability_gate_candidate_metrics.csv', index=False)
(CARD / 'agentic_alpha_meta_label_probability_gate_search_summary.json').write_text(
    json.dumps(search_summary, indent=2, sort_keys=True),
    encoding='utf-8',
)

branch = search_summary['branch_results']['meta_label_probability_gate']
failure_modes = f'''# Failure Modes

- meta_label_probability_gate: raw numeric pass exists ({search_summary['raw_passes_both_count']} rows), but the best drawdown-gated row uses max_exposure={branch['best_drawdown_gate_max_exposure']:.2f}x and trade_count={branch['best_drawdown_gate_trade_count']}, failing the sanity gates.
- High-confidence meta-label events are too sparse to prove a durable strategy over the full 2019-2026 window.
- next_action: require an independent crash-risk signal or longer intraday/order-flow history before reconsidering probability-gated SOXL sizing.
'''
(CARD / 'failure_modes.md').write_text(failure_modes, encoding='utf-8')
print(json.dumps({
    'raw_passes_both_count': search_summary['raw_passes_both_count'],
    'passes_both_count': search_summary['passes_both_count'],
    'branch': branch,
}, indent=2, sort_keys=True))

[76699:MainThread](2026-06-23 18:51:13,630) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[76699:MainThread](2026-06-23 18:51:13,632) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[76699:MainThread](2026-06-23 18:51:13,632) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


{
  "branch": {
    "best_drawdown_gate_available": true,
    "best_drawdown_gate_candidate": "base_dip_LogisticRegression_p0.9_binary_scale100",
    "best_drawdown_gate_max_drawdown_pct": -5.521588885514972,
    "best_drawdown_gate_max_exposure": 100.0,
    "best_drawdown_gate_return_pct": 13104.849328472821,
    "best_drawdown_gate_trade_count": 3,
    "best_return_candidate": "base_dip_LogisticRegression_p0.9_binary_scale100",
    "best_return_max_drawdown_pct": -5.521588885514972,
    "best_return_max_exposure": 100.0,
    "best_return_pct": 13104.849328472821,
    "best_return_trade_count": 3,
    "coverage_end": "2026-06-09",
    "coverage_start": "2024-01-02",
    "goal_baseline_return_pct": 6570.0232821043,
    "grid_rows": 1408,
    "max_drawdown_floor": -20.0,
    "passes_both_count": 0,
    "raw_passes_both_count": 4
  },
  "passes_both_count": 0,
  "raw_passes_both_count": 4
}


In [3]:

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder_id = recorder.id
    R.log_params(
        baseline=baseline.name,
        provider_uri=PROVIDER_URI.as_posix(),
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        max_exposure_sanity=MAX_EXPOSURE_SANITY,
        min_trade_count_sanity=MIN_TRADE_COUNT_SANITY,
        source_url='https://github.com/JacobDu/agentic-alpha',
        metric_rows=len(metrics),
    )
    search_summary['recorder_id'] = recorder_id
    (CARD / 'agentic_alpha_meta_label_probability_gate_search_summary.json').write_text(
        json.dumps(search_summary, indent=2, sort_keys=True),
        encoding='utf-8',
    )
    for path in [
        CARD / 'agentic_alpha_meta_label_probability_gate_candidate_metrics.csv',
        CARD / 'agentic_alpha_meta_label_probability_gate_search_summary.json',
        CARD / 'external_sources.csv',
        CARD / 'failure_modes.md',
    ]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_meta_label_probability_gate')
    recorder_local_dir = Path(recorder.get_local_dir())

for transient_name in ['code_cached.txt', 'code_diff.txt', 'code_status.txt']:
    transient_path = recorder_local_dir / 'artifacts' / transient_name
    if transient_path.exists():
        transient_path.unlink()
recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
search_summary['recorder_artifacts'] = recorder_artifacts
search_summary['recorder_id'] = recorder_id
search_summary['recorder_local_dir'] = recorder_local_dir.as_posix()
(CARD / 'agentic_alpha_meta_label_probability_gate_search_summary.json').write_text(
    json.dumps(search_summary, indent=2, sort_keys=True),
    encoding='utf-8',
)
print(f'recorder_id={recorder_id}')


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[76699:MainThread](2026-06-23 18:51:27,634) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_meta_label_probability_gate_search_v1.


[76699:MainThread](2026-06-23 18:51:27,636) INFO - qlib.workflow - [exp.py:258] - Experiment 661214028368112368 starts running ...


[76699:MainThread](2026-06-23 18:51:27,709) INFO - qlib.workflow - [recorder.py:345] - Recorder 31dd7965ce8546c6b0a353178127c003 starts running under Experiment 661214028368112368 ...


[76699:MainThread](2026-06-23 18:51:27,802) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


recorder_id=31dd7965ce8546c6b0a353178127c003


In [4]:
summary = {
    'experiment': EXPERIMENT,
    'card_type': 'diagnostic_search',
    'promotion_decision': 'not_promoted',
    'as_of': END,
    'data_range': [START, END],
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'market': 'all',
    'region': 'us',
    'baseline': search_summary['baseline'],
    'goal_gate': {
        'max_drawdown_floor': MAX_DRAWDOWN_FLOOR,
        'require_return_above_baseline': True,
        'max_exposure_sanity': MAX_EXPOSURE_SANITY,
        'min_trade_count_sanity': MIN_TRADE_COUNT_SANITY,
    },
    'search_report': search_summary,
    'branch_results': search_summary['branch_results'],
    'external_source_counts': search_summary['external_source_counts'],
    'meta_label_probability_gate': {
        'selected_features': META_LABEL_SOURCE_FIELDS,
        'model_families': META_LABEL_MODEL_FAMILIES,
        'primary_signals': ['base_trend', 'base_dip', 'base_calendar', 'base_any'],
        'raw_numeric_passes_require_sanity': True,
    },
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'model': 'https://qlib.readthedocs.io/en/latest/component/model.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
    },
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start'],
    'workflow_config': 'workflow.yaml',
    'sidecars': {
        'candidate_metrics': 'agentic_alpha_meta_label_probability_gate_candidate_metrics.csv',
        'external_sources': 'external_sources.csv',
        'failure_modes': 'failure_modes.md',
        'search_summary': 'agentic_alpha_meta_label_probability_gate_search_summary.json',
    },
    'artifact_policy': {
        'policy': 'slim_card_recorder_first',
        'workflow_config': 'workflow.yaml',
        'machine_entrypoint': 'summary.json',
        'primary_artifact': f'{EXPERIMENT}.ipynb',
        'kept_sidecars': [
            'agentic_alpha_meta_label_probability_gate_candidate_metrics.csv',
            'agentic_alpha_meta_label_probability_gate_search_summary.json',
            'external_sources.csv',
            'failure_modes.md',
        ],
        'recorder_only_artifacts': {
            'predictions': 'artifacts/pred.pkl',
            'labels': 'artifacts/label.pkl',
            'signals': 'artifacts/signals.pkl',
            'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
            'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
            'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
            'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        },
        'reproducible_from_recorder': True,
    },
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')

branch = search_summary['branch_results']['meta_label_probability_gate']
conclusion = f'''# 结论

- what_ran: 执行 {EXPERIMENT}，用 qlib `D.features` 读取 SOXL/SOXX/VIX，按 meta-labeling 思路训练 LogisticRegression 与 HistGradientBoosting 年度 walk-forward probability gate。
- key_metrics: production baseline return={baseline.return_pct:.2f}%, maxDD={baseline.max_drawdown_pct:.2f}%; raw_passes_both_count={search_summary['raw_passes_both_count']}, sanity_passes_both_count={search_summary['passes_both_count']}。
- regime_window_difference: raw 最佳候选集中在少数高置信 dip 事件，best_drawdown_gate_return={branch['best_drawdown_gate_return_pct']:.2f}%, MaxDD={branch['best_drawdown_gate_max_drawdown_pct']:.2f}%, 但 trade_count={branch['best_drawdown_gate_trade_count']} 且 max_exposure={branch['best_drawdown_gate_max_exposure']:.2f}x。
- promotion_decision: not_promoted；裸数值 pass 来自极端暴露和稀疏交易，未通过 max_exposure<=10x 与 trade_count>=20 的 sanity gate。
- risks: 样本太少、暴露过高、meta-label probability 可能过拟合近期 dip rebound；不能作为生产策略或目标仓位。
- next_action: 引入独立 crash-risk / options / order-flow 数据，或把 probability gate 约束在可交易暴露内重新搜索。
'''
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print(json.dumps({'summary': 'written', 'promotion_decision': summary['promotion_decision']}, indent=2))

{
  "summary": "written",
  "promotion_decision": "not_promoted"
}
